# PharmaFlow Foundation — Session 1.1 Solutions
### Python Fundamentals: Variables, Data Types & Control Flow

These are the instructor solutions for all five exercise tiers. Every cell below has
been run against the real dataset for this session (`adverse_events_100.json`) — the
outputs shown are actual output, not illustrative text.

**Dataset:** 100 real FDA adverse event reports, pulled directly from the OpenFDA
Drug Adverse Events API (`api.fda.gov/drug/event.json`) and curated down to the
fields this session needs (`safety_report_id`, `drug_name`, `reaction`, `serious`,
`patient_age`, `country`, `event_date`). 31 records are missing `patient_age` --
that's not a mistake, it's how the FDA received them; real submitters don't always
report every field, and Session 1.1 is where students first meet that fact.

## Before You Run Anything — Environment Check

This notebook works out the box in VS Code, JupyterLab, or classic Jupyter --
it finds the repo automatically, no matter which folder your notebook tool
happened to start in (different tools default to different folders here,
so we don't rely on that).

Run the cell below first. If it prints an error instead of "You're ready to
go.", stop here and fix your setup before continuing -- every exercise from
this point on depends on the `DATA_FILE` variable it defines.

In [1]:
# --- Setup check: always run this cell first ---
import os

def find_repo_root(marker_folder="data", start=None):
    """
    Different notebook tools start a notebook's kernel in different
    folders -- VS Code, JupyterLab, and classic Jupyter have all been seen
    defaulting to the folder the notebook FILE sits in, not the repo root,
    regardless of which folder you opened in the tool itself.

    Instead of assuming a specific working directory, we search upward
    from wherever we happened to start until we find a folder that
    contains `data` -- that only exists at the repo root, so wherever we
    find it IS the repo root. This makes every path below work the same
    way no matter which tool opened this notebook.
    """
    current = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.isdir(os.path.join(current, marker_folder)):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            raise FileNotFoundError(
                f"Could not find a '{marker_folder}' folder anywhere above "
                f"{os.getcwd()}. Make sure this notebook is somewhere inside "
                f"the pharmaflow-ai repo."
            )
        current = parent


REPO_ROOT = find_repo_root()
DATA_FILE = os.path.join(REPO_ROOT, "data", "raw", "adverse_events_100.json")

if os.path.exists(DATA_FILE):
    print("Repo root found at:", REPO_ROOT)
    print("Data file found at:", DATA_FILE)
    print("You're ready to go.")
else:
    print("Repo root found at:", REPO_ROOT)
    print("Data file NOT found at:", DATA_FILE)
    print()
    print("Fix: confirm adverse_events_100.json was added to data/raw/ at the")
    print("top level of the pharmaflow-ai repo (not inside week01/ or any")
    print("session folder). See: Trainer Guide > Appendix: GitHub & Environment Setup.")

Repo root found at: /workspaces/pharmaflow-ai
Data file found at: /workspaces/pharmaflow-ai/data/raw/adverse_events_100.json
You're ready to go.


## Warm-Up — Six students, two subjects (no FDA data yet)

Every mechanic used for the rest of this session — indexing, comparisons,
loops, `.get()` — on invented data small enough to check by hand first.

In [2]:
# WARM-UP: tiny invented data, no FDA records yet.
# Same toy_scores idea used in the SQL module's Session 3.1 warm-up --
# same shape, different language.

toy_scores = [
    {"student": "Asha", "subject": "Math", "score": 88},
    {"student": "Asha", "subject": "Science", "score": 73},
    {"student": "Ravi", "subject": "Math", "score": 65},
    {"student": "Ravi", "subject": "Science", "score": 91},
    {"student": "Meera", "subject": "Math", "score": 95},
    {"student": "Meera", "subject": "Science", "score": 95},
]

# Indexing: grab the first record
first = toy_scores[0]
print(f"First record: {first['student']} scored {first['score']} in {first['subject']}")

# Slicing: just the first three
first_three = toy_scores[:3]
print(f"First three students: {[r['student'] for r in first_three]}")

# Comparison + boolean logic + control flow, on data small enough to check by eye
for record in toy_scores:
    if record["score"] >= 80:
        print(f"{record['student']} ({record['subject']}): {record['score']} -- HIGH")
    else:
        print(f"{record['student']} ({record['subject']}): {record['score']} -- not high")

# enumerate(): index + value together, without a manual counter
print("\nWith position numbers:")
for i, record in enumerate(toy_scores):
    print(f"  {i}: {record['student']} -- {record['score']}")

# .get() vs [] on a dict
print(f"\nrecord['student'] -> {first['student']}")
print(f"record.get('student') -> {first.get('student')}")
print(f"record.get('grade_level') -> {first.get('grade_level')}")       # key doesn't exist
print(f"record.get('grade_level', 'N/A') -> {first.get('grade_level', 'N/A')}")  # with a default


First record: Asha scored 88 in Math
First three students: ['Asha', 'Asha', 'Ravi']
Asha (Math): 88 -- HIGH
Asha (Science): 73 -- not high
Ravi (Math): 65 -- not high
Ravi (Science): 91 -- HIGH
Meera (Math): 95 -- HIGH
Meera (Science): 95 -- HIGH

With position numbers:
  0: Asha -- 88
  1: Asha -- 73
  2: Ravi -- 65
  3: Ravi -- 91
  4: Meera -- 95
  5: Meera -- 95

record['student'] -> Asha
record.get('student') -> Asha
record.get('grade_level') -> None
record.get('grade_level', 'N/A') -> N/A


## Exercise 1 (Beginner) — Read one record straight

In [3]:
import json

with open(DATA_FILE) as f:
    records = json.load(f)

# Tier 1 - Beginner: pull the first record and report on it
first_event = records[0]

drug = first_event["drug_name"]
reaction = first_event["reaction"]
is_serious = first_event["serious"] == "1"

print(f"Drug: {drug}")
print(f"Reaction reported: {reaction}")
print(f"Serious event: {is_serious}")


Drug: LETAIRIS
Reaction reported: Back pain
Serious event: False


## Exercise 2 (Intermediate) — Scan ten records, flag the serious ones

In [4]:
import json

with open(DATA_FILE) as f:
    records = json.load(f)

# Tier 2 - Intermediate: summarize the first 10 records, flagging serious events
first_ten = records[:10]

for event in first_ten:
    drug = event["drug_name"]
    reaction = event["reaction"]
    serious_code = event["serious"]

    if serious_code == "1":
        flag = "*** SERIOUS ***"
    elif serious_code == "2":
        flag = "non-serious"
    else:
        flag = "UNKNOWN"

    print(f"[{event['safety_report_id']}] {drug:<35} -> {reaction:<46} {flag}")


[10003314] LETAIRIS                            -> Back pain                                      non-serious
[10003316] LETAIRIS                            -> Oedema                                         non-serious
[10003317] BENLYSTA                            -> Angioedema                                     *** SERIOUS ***
[10003319] BENLYSTA                            -> Cholecystectomy                                *** SERIOUS ***
[10003322] LETAIRIS                            -> Tremor                                         non-serious
[10003323] PRADAXA                             -> Fall                                           non-serious
[10003331] MIRENA                              -> Chemical poisoning                             *** SERIOUS ***
[10003332] PRADAXA                             -> Proctalgia                                     non-serious
[10003351] MIRTAZAPINE (UNKNOWN)               -> Aggression                                     *** SERIOUS ***
[10

**Trainer note for this output:** this 10-record slice already shows both branches of
the `if`/`elif` (5 serious, 5 non-serious) and both common real-name patterns --
plain names like `LETAIRIS`, and free-text entries with a manufacturer folded in
like `TACROLIMUS (WATSON LABORATORIES)`. That second pattern is real FDA
free-text data entry, not a formatting bug -- worth calling out before a student
asks. Full-dataset counts are in Exercise 3 (60 serious / 40 non-serious).

## Exercise 3 (Advanced) — A reusable, error-safe filter function

In [5]:
import json

with open(DATA_FILE) as f:
    records = json.load(f)

# Tier 3 - Advanced: a reusable function with error handling + a list comprehension
def filter_serious_events(event_list):
    """Return only the events flagged as serious ('1').

    Defends against a record missing the 'serious' field entirely by skipping
    it and counting it separately, rather than letting the whole script crash.
    In THIS particular file that branch happens not to fire -- every record
    here reports 'serious' -- but production FDA data can and does drop any
    field on any given day, so the defense stays in regardless. (This file's
    actual gap is 'patient_age', missing on 31 records -- see Exercise 4.)
    """
    serious_events = []
    skipped_count = 0

    for event in event_list:
        try:
            if event["serious"] == "1":
                serious_events.append(event)
        except KeyError:
            skipped_count += 1

    return serious_events, skipped_count


serious, skipped = filter_serious_events(records)

# List comprehension: pull just the drug names out of the serious events
serious_drug_names = [event["drug_name"] for event in serious if "drug_name" in event]

print(f"Total records: {len(records)}")
print(f"Serious events: {len(serious)}")
print(f"Records skipped (missing 'serious' field): {skipped}")
print(f"Unique drugs involved in serious events: {len(set(serious_drug_names))}")


Total records: 100
Serious events: 60
Records skipped (missing 'serious' field): 0
Unique drugs involved in serious events: 48


## Exercise 4 (Corporate Scenario) — The 2am ingestion crash, fixed for good

In [6]:
import json

with open(DATA_FILE) as f:
    records = json.load(f)

issue_log = []

def log_issue(record_id, *problems):
    """*args lets us log a record with ONE problem or FIVE, without
    changing the function signature. This is the pattern the corporate
    scenario asks for: the ingestion script must never care in advance
    how many things could be wrong with a record."""
    issue_log.append({"record_id": record_id, "problems": list(problems)})


def clean_record(event):
    """Validate a single record. Returns a cleaned dict, or None if the
    record is unusable. Never raises -- every failure path is caught
    and logged instead, which is the whole point of the 2am-crash fix."""
    record_id = event.get("safety_report_id", "UNKNOWN_ID")
    problems = []

    try:
        drug = event["drug_name"]
    except KeyError:
        problems.append("missing drug_name")
        drug = None

    reaction = event.get("reaction") or None
    if not reaction:
        problems.append("missing/empty reaction")

    age_raw = event.get("patient_age")
    try:
        age = int(age_raw) if age_raw is not None else None
    except (ValueError, TypeError):
        problems.append(f"unparseable patient_age: {age_raw!r}")
        age = None
    if age is None:
        problems.append("missing patient_age")

    serious_code = event.get("serious")
    if serious_code not in ("1", "2"):
        problems.append(f"missing/invalid serious code: {serious_code!r}")

    if problems:
        log_issue(record_id, *problems)

    if drug is None or serious_code not in ("1", "2"):
        return None  # too broken to safely include in the report

    return {
        "record_id": record_id,
        "drug_name": drug,
        "reaction": reaction,
        "serious": serious_code == "1",
    }


def build_summary_report(event_list):
    cleaned = []
    for event in event_list:
        try:
            result = clean_record(event)
            if result is not None:
                cleaned.append(result)
        finally:
            pass  # placeholder for where a real pipeline would close a per-record trace span

    serious_count = sum(1 for r in cleaned if r["serious"])
    reaction_tally = {}
    for r in cleaned:
        reaction_tally[r["reaction"]] = reaction_tally.get(r["reaction"], 0) + 1
    top_reaction = max(reaction_tally, key=reaction_tally.get) if reaction_tally else None

    return {
        "total_records_received": len(event_list),
        "records_usable": len(cleaned),
        "records_quarantined": len(event_list) - len(cleaned),
        "serious_events": serious_count,
        "non_serious_events": len(cleaned) - serious_count,
        "top_reaction_reported": top_reaction,
    }


report = build_summary_report(records)

print("=== PharmaFlow Daily Ingestion Summary ===")
for key, value in report.items():
    print(f"{key}: {value}")

print(f"\n{len(issue_log)} records flagged during cleaning (script did not crash):")
for entry in issue_log[:5]:
    print(f"  {entry['record_id']}: {entry['problems']}")
if len(issue_log) > 5:
    print(f"  ... and {len(issue_log) - 5} more (see full log)")


=== PharmaFlow Daily Ingestion Summary ===
total_records_received: 100
records_usable: 100
records_quarantined: 0
serious_events: 60
non_serious_events: 40
top_reaction_reported: Death

31 records flagged during cleaning (script did not crash):
  10003302: ['missing patient_age']
  10003315: ['missing patient_age']
  10003334: ['missing patient_age']
  10003342: ['missing patient_age']
  10003355: ['missing patient_age']
  ... and 26 more (see full log)


## Exercise 5 (Interview Challenge) — No imports allowed

In [7]:
import json

with open(DATA_FILE) as f:
    records = json.load(f)

# Tier 5 - Interview Challenge: no imports allowed (tests raw fundamentals,
# not library knowledge -- this is exactly how a junior DE screen is run)
def most_reported_drugs(event_list):
    """Return the drug name(s) with the most reported events.
    Returns a list because ties are a real possibility and silently
    picking one winner would be a bug, not a simplification."""
    counts = {}
    for event in event_list:
        drug = event.get("drug_name")
        if drug is None:
            continue
        counts[drug] = counts.get(drug, 0) + 1

    if not counts:
        return [], 0

    highest = 0
    for count in counts.values():
        if count > highest:
            highest = count

    winners = []
    for drug, count in counts.items():
        if count == highest:
            winners.append(drug)

    return sorted(winners), highest


top_drugs, top_count = most_reported_drugs(records)
print(f"Most-reported drug(s) ({top_count} events each): {top_drugs}")

# Self-check against a hand-verifiable tiny case, the way an interviewer would push
test_data = [
    {"drug_name": "A"}, {"drug_name": "B"}, {"drug_name": "A"},
    {"drug_name": "B"}, {"drug_name": "C"},
]
assert most_reported_drugs(test_data) == (["A", "B"], 2), "tie-handling is broken"
print("Tie-handling self-check passed.")


Most-reported drug(s) (3 events each): ['CLARITIN CHEWABLE TABLETS', 'DEPAKOTE', 'GILENYA', 'GLIVEC', 'HUMIRA', 'JAKAFI', 'LETAIRIS', 'LYRICA', 'MIRENA', 'NEULASTA', 'NEXPLANON']
Tie-handling self-check passed.


---
### Grading notes
- With this real dataset, Exercise 3's `except KeyError` branch legitimately does
  **not** fire (`serious` is present on all 100 records) -- `skipped == 0` is the
  correct answer, not a bug. Grade the *code structure* (a real try/except around
  the field access) rather than expecting the branch to trigger. Exercise 4 is
  where a real gap actually fires: 31 records are missing `patient_age`, and a
  correct solution logs all 31 without crashing or quarantining them.
- Exercise 5's tie-handling is the single most commonly-missed requirement -- most
  first attempts return only one drug name even when several are tied. With this
  dataset it's *not* a hypothetical: 11 drugs genuinely tie at 3 events each, so
  a solution that silently returns one winner is visibly wrong, not just
  technically incomplete. The assert block is there so students can self-check
  before submitting.